# 01 - Scraping Data Inflasi Indonesia
Notebook ini berisi proses pengambilan data dari berbagai sumber (IMF, FRED, TradingEconomics) secara otomatis.

In [ ]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# Add src to path
sys.path.append('../')

load_dotenv('../.env')
os.makedirs('../data/raw', exist_ok=True)

## 1. IMF Scraping (Inflasi & Reserves)
Menggunakan REST API publik IMF.

In [ ]:
from src.scraper.imf_scraper import scrape_imf

print("Scraping IMF data...")
imf_inf = scrape_imf("PCPIPCH")
if not imf_inf.empty:
    imf_inf.to_csv('../data/raw/imf_inflation.csv', index=False)
    display(imf_inf.head())

## 2. FRED Scraping (Macro Indicators)
Membutuhkan API Key di file `.env`.

In [ ]:
from src.scraper.fred_scraper import scrape_fred, FRED_SERIES

all_fred = []
for series_id, col_name in FRED_SERIES.items():
    print(f"Scraping FRED: {series_id} -> {col_name}")
    df = scrape_fred(series_id, start_date="2010-01-01")
    if not df.empty:
        df.rename(columns={series_id.lower(): col_name}, inplace=True)
        all_fred.append(df)

if all_fred:
    from functools import reduce
    fred_combined = reduce(lambda l, r: pd.merge(l, r, on='date', how='outer'), all_fred)
    fred_combined.to_csv('../data/raw/fred_data.csv', index=False)
    display(fred_combined.head())
else:
    print("No FRED data collected. Check your API Key in .env")

## 3. TradingEconomics Scraping

In [ ]:
from src.scraper.te_scraper import scrape_te_inflation

te_df = scrape_te_inflation()
if not te_df.empty:
    te_df.to_csv('../data/raw/te_inflation.csv', index=False)
    display(te_df.head())
else:
    print("Scraping TE failed or returned empty.")